# Neural Networks and Deep Learning, MDS HSE

## Homework 1. Fully Connected Neural Networks. Bonus.

### General Information

### Grading and Penalties

The maximum possible grade for the assignment without bonuses is 10 points. Submitting the work after the hard deadline is not allowed.

Submitting after the soft deadline incurs a penalty of -1 point per day. Twice per semester (two modules), students are allowed to use an extension and submit by the hard deadline without penalty.

The assignment must be completed individually. “Similar” solutions will be considered plagiarism, and all involved students (including those whose work was copied) will receive no more than 0 points for the assignment. If you find a solution (or part of it) to any task from an open source, you must include a link to that source in a separate section at the end of your work (most likely you won’t be the only one who found it, so providing the link helps avoid suspicion of plagiarism).

Inefficient code implementation may negatively affect your grade. The grade may also be reduced for poorly readable code and poorly formatted plots. All answers must be accompanied by either code or comments explaining how they were obtained.

Use of generative models is allowed under the following conditions:
- The amount of code generated by such models does not exceed 30% of the total.
- You specify the model used and the prompt.
- At the end of your work, you include a **reflection on your experience using generative AI for this homework:  
  Describe how often you had to fix the code yourself or ask the model to correct something. Was it faster than writing the code on your own?

If these requirements are not met, the assignment will not be graded, and the maximum possible score is 0 points.

### About the task

In this task, you will implement your own framework for training neural networks based on `numpy`. The framework's interface will closely resemble PyTorch, so you will get a little insight into how everything works internally. The `modules` directory contains files with framework templates, and `tests` contains tests to verify the correctness of your implementations.

The cell below allows you to reload Python modules that you have changed after importing, without having to restart your notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import modules as mm
import numpy as np
import tests
from tqdm.notebook import tqdm

## 0. Automatic Differentiation

The most important object in our framework will be the layer abstraction (the `Module` class), which is implemented in the `modules/base.py` file. Before writing your code, familiarize yourself with the implementation of the `Module` class. Each layer must support two operations: forward pass, which takes the output of the previous layer and computes the layer function, and backward pass, which takes the output of the previous layer and the derivative of its output, and returns the derivative of the input, updating the gradient of its parameters along the way. Let's recall the general scheme once again. Let $f(x, w)$ be our layer function, which depends on the input $x$ and the parameters in $w$, and $\ell$ be the loss function, whose gradient we are interested in. Then:

- Forward pass:
- 
$$y = f(x, w)$$

- Backward pass:

$$\frac{d\ell}{dx} = \frac{d\ell}{dy} \cdot \frac{df(x, w)}{dx}$$

$$\frac{d\ell}{dw} = \frac{d\ell}{dy} \cdot \frac{df(x, w)}{dw}$$

Thus, when passing forward, $x$ is passed to the layer, and when passing backward, $x$ and $\frac{d\ell}{dy}$ are passed. In addition, each layer saves its output when passing forward so that it can then be passed to the next layer when passing backward. Accordingly, the base class `Module` implements the functions `forward` (or its alias, the method `__call__`, similar to how it is done in PyTorch) and `backward`. In addition, the templates contain some utility functions, including `train` and `eval`, which change the layer mode. All layers that you need to implement will inherit from the `Module` class. In them, you will need to implement the `compute_output`, `compute_grad_input`, and `update_grad_parameters` methods (if the layer has trainable parameters). For details, refer to the doc strings in the templates. We will implement layers similar to the corresponding layers from PyTorch, so you can refer to the documentation to clarify the values of the layer parameters. For your convenience, we provide tests for debugging implementations; your solution must pass them (points are awarded for passed tests). If you encounter difficulties, we recommend debugging the code by comparing your implementation with modules from PyTorch.

**Important:** we want to get the same behavior as PyTorch, so if you make several backward passes without calling the `zero_grad` function, the gradients from all backward passes should be summed up. This means that in the `update_grad_parameters` function, you need to add the new gradient to the existing one, rather than overwriting it.

Please note that all your function and gradient calculations must be **vectorized**, i.e., they must only include operations on `numpy`/`scipy` and **no Python loops** (or wrappers over them from the specified libraries). Special cases where loops are allowed will be indicated separately.

## 1. Linear Layer (1 point)

- Prototype: [nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear)
- Location: `modules.layers.Linear`

From now on, we will assume that the input of the neural network $x$ has a size of $B \times N$, where $B$ is the mini-batch size and $N$ is the dimension. The layer function looks like this:

$$
y = x \, W^T + b,
$$

where $W \in \mathbb{R}^{M \times N}, b \in \mathbb{R}^M$. Thus, the output of the layer has size $B \times M$.

In [ ]:
tests.test_linear()

## 2. Batch normalization (2.5 points)

- Prototype: [nn.BatchNorm1d](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html#torch.nn.BatchNorm1d)
- Location: `modules.layers.BatchNormalization`

Batch normalization is the first layer that works differently in train and eval modes.

**Train mode:**

1. For each input coordinate, we calculate statistics for the mini-batch ($x_i \in \mathbb{R}^N$ — one object in the mini-batch):
   
$$
\mu = \frac{1}{B} \sum_{i=1}^B x_i, \quad\quad \mu \in \mathbb{R}^N \\
\sigma^2 = \frac{1}{B} \sum_{i=1}^B (x_i - \mu)^2, \quad\quad \sigma^2 \in \mathbb{R}^N
$$

Please note that a **biased** estimate of variance is used here (i.e., we divide the sum of squares of deviations by $B$, not by $B-1$).

2. We normalize the input taking into account statistics:

$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}}
$$

3. Apply affine transformation to the normalized input (if `affine = True`), element-wise multiplication. This will be the output of the layer.

$$
y_i = \hat{x}_i * w + b, \quad\quad w, b \in \mathbb{R}^N
$$

4. Update the running statistics of the layer:

$$
\text{running mean} = (1 - \text{momentum}) \cdot \text{running mean} + \text{momentum} \cdot \mu \\
\text{running var} = (1 - \text{momentum}) \cdot \text{running var} + \text{momentum} \cdot \frac{B}{B - 1}
\cdot \sigma^2
$$

Here, renormalization of $\sigma^2$ is necessary to update the running variance with an **unbiased** estimate (this is implemented in PyTorch in exactly the same way).

The layer parameters that are updated by the gradient are only $w$ (`weight`) and $b$ (`bias`), but not `running mean` and `running var`.

**Eval mode:**

1. Normalize the input using running statistics:

$$
\hat{x}_i = \frac{x_i - \text{running mean}}{\sqrt{\text{running var} + \varepsilon}}
$$

2. Apply the affine transformation to the normalized input:

$$
y_i = \hat{x}_i * w + b
$$

**Note**

- Make sure that backward pass works correctly for both train and eval modes.
- Save intermediate calculations during forward pass to reuse them during backward pass.
- It is highly likely that you will not get the correct implementation on the first try. Don't despair; the author of the task also spent many hours before this module worked. If you feel like you've reached a dead end, no one is stopping you from using Google, but don't forget to cite the sources you use.

In [ ]:
tests.test_bn()

## 3. Dropout (1 point)

- Prototype: [nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html#torch.nn.Dropout)
- Location: `modules.layers.Dropout`

Dropout is another layer whose behavior differs in train and eval modes. The behavior of the layer is controlled by the parameter $p$ — the probability of zeroing the input coordinate.

**Train mode:**

Let $m$ denote a binary mask that has the same size as the input $x$. The mask is generated according to the rule $m_{ij} \sim \text{Bernoulli}(1-p)$. At the same time, a new mask is generated with each new forward pass (i.e., it is not fixed). Layer function (element-wise multiplication):

$$
y = \frac{1}{1-p} m * x
$$

Normalization to $1-p$ is necessary so that the average value of the input neurons does not change.

**Eval mode:**

Everything is extremely simple here: the layer input does not change at all $y=x$.

**Note**

- Make sure that the backward pass works correctly for both train and eval modes.

In [ ]:
tests.test_dropout()

## 4. Activation Functions (1.5 points)

### ReLU

- Prototype: [nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
- Location: `modules.activations.ReLU`

Layer function:

$$
y = \max(x, 0)
$$

### Sigmoid

- Prototype: [nn.Sigmoid](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html?highlight=nn%20sigmoid#torch.nn.Sigmoid)
- Location: `modules.activations.Sigmoid`

Layer function:

$$
y = \frac{1}{1 + e^{-x}}
$$

### Softmax

- Prototype: [nn.Softmax](http://bit.ly/get3a)
- Location: `modules.activations.Softmax`

Layer function:

$$
y_{ij} = \frac{\exp(x_{ij})}{\sum_{k=1}^{N} \exp(x_{ik})}
$$

### LogSoftmax

- Prototype: [nn.LogSoftmax](https://pytorch.org/docs/stable/generated/torch.nn.LogSoftmax.html?highlight=log%20softmax#torch.nn.LogSoftmax)
- Location: `modules.activations.LogSoftmax`

Layer function:

$$
y_{ij} = \log \left(\frac{\exp(x_{ij})}{\sum_{k=1}^{N} \exp(x_{ik})}\right)
$$

**Note to the hostess**

- Use functions from `scipy.special`
- Implementing `LogSoftmax` as the logarithm of the `Softmax` module is a bad idea

In [ ]:
tests.test_activations()

## 5. Sequential Container (1 point)

- Prototype: [nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
- Location: `modules.layers.Sequential`

A wrapper container that applies layers sequentially.

**Important:** Forward and backward module cycles are allowed here.

In [ ]:
tests.test_sequential()

## 6. Loss functions (1 point)

Loss functions differ from all other modules in that they are the sink of the computational graph (i.e., there are no outgoing operations from them). This means that the backward pass starts from them, so the interface of the `compute_grad_input` function looks different: instead of the module input and output derivative, the function receives the neural network prediction (the derivative of which we are interested in to start the backward pass) and the target variable. The base class for all loss functions is `modules.base.Criterion`.

### MSE

- Prototype: [nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
- Location: `modules.criterions.MSELoss`

Let $f \in \mathbb{R}^{B\times N}$ be the neural network prediction and $y \in \mathbb{R}^{B\times N}$ be the target variable. The loss function looks like this:

$$
\ell(f, y) = \frac{1}{BN} \sum_{i=1}^B \sum_{j=1}^N (f_{ij} - y_{ij})^2
$$

### Cross Entropy

- Prototype: [nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- Location: `modules.criterions.CrossEntropyLoss`

Cross-entropy is a loss function used for training classifiers.  
Let $f \in \mathbb{R}^{B\times C}$ (where $C$ is the number of classes) be the neural network’s prediction (these are the so-called *logits*, usually the outputs of a linear layer without activation, so they can be any real numbers), and let $y \in \{1, \dots, C\}^B$ be the target variable (the class index for each corresponding object).  
The loss function is computed as:

$$
p_{ic} = \frac{\exp(f_{ic})}{\sum_{k=1}^C \exp(f_{ik})}
$$

$$
\ell(f, y) = -\frac{1}{B} \sum_{i=1}^B \sum_{c=1}^C [c = y_i] \log p_{ic}
$$

Here, $p_{ic}$ is the probability of class $c$ for object $i$ predicted by the neural network.

**Important:** Calculating Softmax and then its logarithm is a numerically unstable operation. Use the LogSoftmax layer.

In [ ]:
tests.test_criterions()

## 7. Optimizers (1.5 points)

An optimizer is a helper class that updates the weights of a neural network during gradient descent using stored parameter gradients. The base class is `modules.base.Optimizer`. The PyTorch documentation provides pseudocode describing the algorithms; we recommend referring to it.

**Important:** a loop over parameters and gradients is allowed here (see templates)

### SGD

- Prototype: [torch.optim.SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html)
- Location: `modules.optimizers.SGD`

### Adam

- Prototype: [torch.optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#torch.optim.Adam)
- Location: `modules.criterions.CrossEntropyLoss`

In [ ]:
tests.test_optimizers()

## 8. DataLoader (0.5 points)

- Prototype: [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
- Location: `modules.dataloader.DataLoader`

The last thing we need to implement is DataLoader, which shuffles the data once per epoch (if necessary) and forms mini-batches from it. Technically, this will be a Python iterator. Here is a brief [guide](https://stackoverflow.com/questions/19151/how-to-build-a-basic-iterator) on how to write an iterator.

Please note that your implementation must be able to work with both a one-dimensional array of the target variable (with the form `(num_samples, )` — this will make it easier to train the neural network on cross-entropy) and a two-dimensional version (with the form `(num_samples, 1)` — correspondingly, on MSE).

In [ ]:
tests.test_dataloader()

## Putting it all together

If you've done everything correctly, the following piece of code with neural network training should work.

In [ ]:
np.random.seed(42)
X_train = np.random.randn(2048, 8)
X_test = np.random.randn(512, 8)
y_train = np.sin(X_train).sum(axis=1, keepdims=True)
y_test = np.sin(X_test).sum(axis=1, keepdims=True)

train_loader = mm.DataLoader(X_train, y_train, batch_size=64, shuffle=True)
test_loader = mm.DataLoader(X_test, y_test, batch_size=64, shuffle=False)

model = mm.Sequential(
    mm.Linear(8, 32),
    mm.BatchNormalization(32),
    mm.ReLU(),
    mm.Linear(32, 64),
    mm.Dropout(0.25),
    mm.Sigmoid(),
    mm.Linear(64, 1),
)
optimizer = mm.Adam(model, lr=1e-2)
criterion = mm.MSELoss()

In [ ]:
num_epochs = 100
pbar = tqdm(range(1, num_epochs + 1))

for epoch in pbar:
    train_loss, test_loss = 0.0, 0.0

    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        model.backward(X_batch, criterion.backward(predictions, y_batch))
        optimizer.step()

        train_loss += loss * X_batch.shape[0]

    model.eval()
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        test_loss += loss * X_batch.shape[0]

    train_loss /= train_loader.num_samples()
    test_loss /= test_loader.num_samples()
    pbar.set_postfix({"train loss": train_loss, "test loss": test_loss})